# Apolos: YouVersion + canvas-aware AI

This notebook provides reproducible evidence for the Apolos hackathon preparation kit. It uses the public Apolos backend to inspect the licensed YouVersion catalog. The product's AI assistant runs server-side through LLPhant and DeepSeek; no provider secret is exposed here.

In [ ]:
import requests

APOLOS_API = "https://apolos.io"
catalog = requests.get(
    f"{APOLOS_API}/api/youversion/versions",
    params={"language": "en", "popular": 0},
    timeout=20,
)
catalog.raise_for_status()
bibles = catalog.json()["data"]
print(f"Licensed English editions returned: {len(bibles)}")
[(b["id"], b["abbreviation"], b["localized_title"]) for b in bibles[:10]]

In [ ]:
expected = {"NIV11", "NIrV", "NASB2020", "AMP"}
available = {b["abbreviation"] for b in bibles}
print("Expected licensed editions present:", sorted(expected & available))
assert expected <= available

## Production architecture

The React/Tauri client calls Laravel. Laravel proxies and caches only successful YouVersion responses. The authenticated canvas assistant is provider-independent through `App\\Contracts\\LlmClient`; its current implementation uses LLPhant with the configured DeepSeek model. Existing verified-email checks, endpoint throttles, and per-user AI budgets apply before inference.

**Eligibility note:** the challenge text supplied to the project requires Gloo AI Studio. Apolos intentionally does not use Gloo, so this remains an explicit submission blocker rather than a claimed integration.